# DeBERTa-v3 Test Inference and Submission

This notebook restores the completed DeBERTa-v3 LoRA model, reproduces its validation score, predicts the test set in its original order, and creates `q2_submission.csv`.

## Kaggle inputs

Attach the processed `balanced-50000` dataset and the `deberta-v3-keras` Kaggle model before running the notebook.

In [ ]:
import os, subprocess, sys
os.environ['KERAS_BACKEND'] = 'tensorflow'
try:
    import keras_hub
except ImportError:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'keras-hub'])
    import keras_hub
import keras
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import tensorflow as tf
print('TensorFlow:', tf.__version__)
print('Keras:', keras.__version__)
print('KerasHub:', keras_hub.__version__)

## Configuration

In [ ]:
from pathlib import Path
INPUT_ROOT = Path('/kaggle/input')
OUTPUT_DIR = Path('/kaggle/working/deberta_v3_test_inference')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
NUM_CLASSES = 5
BATCH_SIZE = 32
EXPECTED_TEST_ROWS = 20_000
LORA_RANK = 16
EXPECTED_MODEL_PARAMETERS = 187_989_509
EXPECTED_VALIDATION_MICRO_F1 = 0.6757
VALIDATION_SCORE_TOLERANCE = 0.002
EXPLICIT_TEST_PATH = '/kaggle/input/datasets/maslri/balanced-50000/bert_balanced_50000_per_class/test.csv'
EXPLICIT_KERAS_MODEL_PATH = None
EXPLICIT_LORA_ADAPTER_PATH = None
EXPLICIT_PRESET_PATH = None
keras.mixed_precision.set_global_policy('mixed_float16')
gpus = tf.config.list_physical_devices('GPU')
if not gpus:
    raise RuntimeError('No GPU detected. Enable a Kaggle GPU accelerator.')
print('GPUs:', gpus)
print('Mixed precision:', keras.mixed_precision.global_policy())

## Locate the data and model

The model files are discovered by artifact name, so the notebook does not depend on a Kaggle model version number.

In [ ]:
def one_match(pattern, explicit=None, required=True):
    if explicit:
        path = Path(explicit)
        if not path.exists():
            raise FileNotFoundError(path)
        return path
    matches = sorted(INPUT_ROOT.rglob(pattern))
    if len(matches) == 1:
        return matches[0]
    if not matches and not required:
        return None
    raise FileNotFoundError(f'Expected one {pattern}; found {matches}')

TEST_PATH = one_match('test.csv', EXPLICIT_TEST_PATH)
KERAS_MODEL_PATH = one_match('deberta_v3_rating_classifier.keras', EXPLICIT_KERAS_MODEL_PATH, False)
LORA_PATH = one_match('deberta_v3_lora_adapters.lora.h5', EXPLICIT_LORA_ADAPTER_PATH)
PRESET_PATH = one_match('deberta_v3_rating_classifier_preset', EXPLICIT_PRESET_PATH, False)
if KERAS_MODEL_PATH is None and PRESET_PATH is None:
    raise FileNotFoundError('No saved DeBERTa-v3 model or preset was found.')
print('Test:', TEST_PATH)
print('Keras model:', KERAS_MODEL_PATH)
print('LoRA adapter:', LORA_PATH)
print('Preset fallback:', PRESET_PATH)

## Load the test set and restore the model

In [ ]:
test_df = pd.read_csv(TEST_PATH, low_memory=False)
test_rows = len(test_df)
test_df['_row_id'] = np.arange(test_rows, dtype='int32')
assert 'overall' not in test_df.columns
assert 'model_input' in test_df.columns and test_df['model_input'].notna().all()
assert test_rows == EXPECTED_TEST_ROWS
test_text = test_df['model_input'].astype(str).to_numpy()
test_ds = tf.data.Dataset.from_tensor_slices(test_text).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

model = None
if KERAS_MODEL_PATH is not None:
    try:
        model = keras.models.load_model(KERAS_MODEL_PATH, compile=False)
        print('Loaded full .keras model.')
    except Exception as error:
        print('The .keras file could not be loaded:', repr(error))
if model is None and PRESET_PATH is not None:
    model = keras_hub.models.DebertaV3TextClassifier.from_preset(str(PRESET_PATH), num_classes=NUM_CLASSES, activation=None)
    print('Loaded saved KerasHub preset.')
if model is None:
    raise RuntimeError('Could not restore the saved model.')
params_before_lora = int(sum(np.prod(v.shape) for v in model.weights))
if params_before_lora < EXPECTED_MODEL_PARAMETERS:
    model.backbone.enable_lora(rank=LORA_RANK)
model.backbone.load_lora_weights(LORA_PATH)
total_params = int(sum(np.prod(v.shape) for v in model.weights))
assert total_params == EXPECTED_MODEL_PARAMETERS, (total_params, EXPECTED_MODEL_PARAMETERS)
print('Restored total parameters:', f'{total_params:,}')
print('Sequence length:', model.preprocessor.sequence_length)

## Reproduce the validation score

A score close to `0.6757` confirms that the correct model, adapters, tokenizer, and processed dataset are attached.

In [ ]:
valid_paths = []
for path in INPUT_ROOT.rglob('validation.csv'):
    try:
        if {'overall', 'model_input'}.issubset(pd.read_csv(path, nrows=2).columns):
            valid_paths.append(path)
    except Exception:
        continue
if len(valid_paths) != 1:
    raise FileNotFoundError(f'Expected one validation.csv; found {valid_paths}')
valid_df = pd.read_csv(valid_paths[0], usecols=['overall', 'model_input'], low_memory=False)
assert len(valid_df) == 10_000
valid_ds = tf.data.Dataset.from_tensor_slices(valid_df['model_input'].astype(str).to_numpy()).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
valid_logits = np.asarray(model.predict(valid_ds, verbose=1), dtype='float32')
valid_pred = np.argmax(valid_logits, axis=1) + 1
valid_f1 = float(np.mean(valid_pred == valid_df['overall'].to_numpy()))
print('Reproduced validation micro F1:', valid_f1)
assert abs(valid_f1 - EXPECTED_VALIDATION_MICRO_F1) <= VALIDATION_SCORE_TOLERANCE
print('Validation reproduction: PASSED')

## Predict and inspect the test set

In [ ]:
logits = np.asarray(model.predict(test_ds, verbose=1), dtype='float32')
assert logits.shape == (test_rows, NUM_CLASSES) and np.isfinite(logits).all()
probs = tf.nn.softmax(logits, axis=-1).numpy()
predictions = np.argmax(probs, axis=1) + 1
confidence = np.max(probs, axis=1)
entropy = -np.sum(probs * np.log(np.clip(probs, 1e-9, 1.0)), axis=1)
distribution = pd.Series(predictions).value_counts().sort_index().reindex(range(1, 6), fill_value=0).to_frame('count')
distribution['percentage'] = (distribution['count'] / test_rows * 100).round(2)
display(distribution)
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
sns.barplot(x=distribution.index, y=distribution['count'], ax=axes[0], color='steelblue')
sns.histplot(confidence, bins=50, ax=axes[1], color='darkorange')
axes[0].set(title='Predicted test rating distribution', xlabel='Rating', ylabel='Rows')
axes[1].set(title='Maximum predicted probability', xlabel='Confidence')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'test_prediction_diagnostics.png', dpi=160)
plt.show()

## Save predictions and submission

In [ ]:
details = pd.DataFrame({'row_id': test_df['_row_id'], 'predicted': predictions, 'confidence': confidence, 'entropy': entropy})
for i in range(NUM_CLASSES):
    details[f'prob_rating_{i + 1}'] = probs[:, i]
submission = pd.DataFrame({'predicted': predictions})
assert details['row_id'].eq(np.arange(test_rows)).all()
assert submission.shape == (test_rows, 1) and submission['predicted'].between(1, 5).all()
submission.to_csv(OUTPUT_DIR / 'q2_submission.csv', index=False)
details.to_csv(OUTPUT_DIR / 'test_predictions_detailed.csv', index=False)
distribution.to_csv(OUTPUT_DIR / 'test_prediction_distribution.csv', index_label='predicted')
analysis_cols = [c for c in ['reviewText', 'summary', 'model_input'] if c in test_df.columns]
low_conf = details.nsmallest(50, 'confidence').merge(test_df[['_row_id'] + analysis_cols], left_on='row_id', right_on='_row_id').drop(columns='_row_id')
low_conf.to_csv(OUTPUT_DIR / 'lowest_confidence_examples.csv', index=False)
print('Saved:', OUTPUT_DIR / 'q2_submission.csv')
display(submission.head())

## Output files

Submit `/kaggle/working/deberta_v3_test_inference/q2_submission.csv`. The detailed predictions and diagnostics are only for analysis.